In [1]:
from google.colab import files
uploaded = files.upload()   # select both CSVs

Saving train_split.csv to train_split.csv
Saving test_split.csv to test_split.csv


In [3]:
import pandas as pd
train_df = pd.read_csv("train_split.csv")
test_df = pd.read_csv("test_split.csv")
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

(155430, 2) (38446, 2)
label
1    10476
2     8835
3    17176
4    32196
5    86747
Name: count, dtype: int64


In [4]:
!pip install -q transformers datasets accelerate


In [5]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

GPU: True Tesla T4


In [6]:
# 50k subsample — full 155k is ~50 min on a free T4 and risks disconnection
train_sub = train_df.sample(50000, random_state=42).reset_index(drop=True)

# Labels must be 0-indexed for the model
train_sub["y"] = train_sub["label"] - 1
test_df["y"] = test_df["label"] - 1

print(train_sub.shape, test_df.shape)

(50000, 3) (38446, 3)


In [7]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_sub[["text", "y"]].rename(columns={"y": "labels"}))
test_ds = Dataset.from_pandas(test_df[["text", "y"]].rename(columns={"y": "labels"}))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
print(train_ds)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/38446 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 50000
})


In [8]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=5)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "macro_f1": f1_score(p.label_ids, preds, average="macro"),
        "mae": mean_absolute_error(p.label_ids, preds),
    }

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    fp16=True,
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Mae
1,0.795883,0.757449,0.693778,0.540094,0.382146
2,0.672991,0.747751,0.699865,0.544535,0.373381


TrainOutput(global_step=3126, training_loss=0.7598391512762791, metrics={'train_runtime': 762.7592, 'train_samples_per_second': 131.103, 'train_steps_per_second': 4.098, 'total_flos': 6594000325257600.0, 'train_loss': 0.7598391512762791, 'epoch': 2.0})

In [9]:
from sklearn.metrics import classification_report, confusion_matrix
preds = np.argmax(trainer.predict(test_ds).predictions, axis=1) + 1
print(classification_report(test_df["label"], preds, digits=3))
print(confusion_matrix(test_df["label"], preds))


              precision    recall  f1-score   support

           1      0.646     0.674     0.660      2771
           2      0.381     0.224     0.282      2217
           3      0.440     0.493     0.465      4224
           4      0.517     0.409     0.457      7700
           5      0.824     0.897     0.859     21534

    accuracy                          0.700     38446
   macro avg      0.562     0.539     0.545     38446
weighted avg      0.682     0.700     0.687     38446

[[ 1868   352   388    55   108]
 [  622   496   845   133   121]
 [  258   363  2081  1017   505]
 [   66    64  1031  3151  3388]
 [   76    26   380  1741 19311]]


### Transformer vs TF-IDF

| Model | Accuracy | Macro F1 | MAE |
|---|---|---|---|
| Majority class | 0.5601 | 0.1436 | 0.8813 |
| TF-IDF + Linear SVM | 0.6241 | 0.4295 | 0.5532 |
| TF-IDF + Logistic Regression | 0.6460 | 0.4418 | 0.5296 |
| TF-IDF + LogReg (balanced) | 0.5786 | 0.4536 | 0.6096 |
| **DistilBERT (fine-tuned)** | **0.6999** | **0.5445** | **0.3734** |

DistilBERT improves accuracy by 5.4 points, macro F1 by 0.10, and reduces
MAE by 29% — from 0.53 to 0.37 stars. The comparison is conservative: the
transformer was fine-tuned on a 50,000-row subsample due to compute
constraints, against 155,430 rows for the TF-IDF models.

**The gain is concentrated where it matters.** TF-IDF predicted 569 actual
1-star reviews as 5-star; DistilBERT predicts 76 — an 87% reduction in the
most costly error type. Class 3, the ambivalent middle identified during
exploration, improved from 0.275 recall to 0.493.

**Class 2 remains the weakest.** Recall improved from 0.100 to 0.224 but it
is still the hardest class: it is the rarest at 5.7% and sits between two
neighbours, so most errors on it go to class 1 or 3 rather than to 5.

This is the one project in the portfolio where added model complexity
clearly earns its place. Across credit risk, customer value and property
forecasting, the simpler specification matched or beat the complex one.
Here a 66-million-parameter transformer requiring GPU training beats
TF-IDF decisively, because the task depends on context and negation rather
than keyword presence.

In [11]:
trainer.save_model("distilbert_review_rating")
tok.save_pretrained("distilbert_review_rating")

# Zip and download to keep with the project
!zip -r distilbert_model.zip distilbert_review_rating
from google.colab import files
files.download("distilbert_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: distilbert_review_rating/ (stored 0%)
  adding: distilbert_review_rating/model.safetensors (deflated 8%)
  adding: distilbert_review_rating/config.json (deflated 54%)
  adding: distilbert_review_rating/training_args.bin (deflated 54%)
  adding: distilbert_review_rating/tokenizer.json (deflated 71%)
  adding: distilbert_review_rating/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 6.5 DistilBERT — fine-tuned
Trained in Google Colab on a T4 GPU. Local CPU training would take several
hours; GPU training takes 13 minutes for 2 epochs on a 50,000-row
subsample. Results are reproduced here; the training script is in
`notebooks/distilbert_colab.ipynb`.